# District-to-District Summary

**Purpose:** Summarize the District-to-District (D2D) tables for the survey trips by category.

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [ ]:
## Data Preparation and Summary
import pandas as pd
import numpy as np

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df = pd.read_csv(interim_dir + "Interim_2023_OnBoardSurvey_Dataset_2.csv")

In [3]:
taz_dis = pd.read_csv(data_dir + "2023OBS_wTAZ_Districts.csv")
taz_dis = taz_dis.drop(columns = ['DATE_COMPLETED', 'ROUTE_DIRECTION[Code]', 'ROUTE_DIRECTION'])

### Merge District

In [4]:
# Append production and attraction TAZ and District
df = pd.merge(df, taz_dis, on='ID', how='left')

In [5]:
# Creating production and attraction districts
df['DIST_PRODUCTION'] = np.where(df['PA_FLAG']==1, df['DIST_origin'], df['DIST_destination'])
df['DIST_ATTRACTION'] = np.where(df['PA_FLAG']==1, df['DIST_destination'], df['DIST_origin'])

## Summary
### Table 6 and Appendix A in Report

In [6]:
# Creating 30 * 30 district index
district_long = pd.DataFrame({
    'DIST_PRODUCTION': np.sort(np.tile(range(1,31),30)),
    'DIST_ATTRACTION': np.tile(range(1,31),30)})

top_district_long = district_long

In [7]:
project_trips = df[df['TRIPS_ON_PROJECT']=='Yes']

In [ ]:
project_trips['DIST_PRODUCTION'] = project_trips['DIST_PRODUCTION'].replace(-1, 30) # District "Other"
project_trips['DIST_ATTRACTION'] = project_trips['DIST_ATTRACTION'].replace(-1, 30)

In [9]:
# Creating D2D tables in long format
# Generating CSV files in Loops: by trip purpose, by auto ownership, by mode of access

data_table = project_trips # specify dataframe

# List of variables to be summarized by
var_list = ['TRIP_PURPOSE_AGG','AUTO_OWNERSHIP','ACCESS_MODE_PA']

# List of variables used as index in pivot table
index_vars = ['DIST_PRODUCTION', 'DIST_ATTRACTION']

for var in var_list:
    # Create pivot table with weighted sum
    D2D_table = pd.pivot_table(data_table,
                               index=index_vars,
                               columns=var,
                               values='REWEIGHTED_LINKED',  # Used weights for aggregation
                               aggfunc='sum',     # Calculate sum of weights
                               fill_value=0)
    
    # Reset index to make 'DIST_boarding' and 'DIST_alighting' columns
    D2D_table = D2D_table.reset_index()

    D2D_table_long_format = pd.merge(district_long, D2D_table, how='left', on = ['DIST_PRODUCTION', 'DIST_ATTRACTION'])

    D2D_table_long_format.fillna(0, inplace=True)

    # Export tables 
    D2D_table_long_format.to_csv(output_dir + f"D2D_TRIPS_ON_PROJECT_{var}_weighted.csv", index=False)
    
    # Display the pivot table for the current variable
    print(f"Pivot Table for {var}:")
    print(D2D_table_long_format)
    print("\n---\n")  # Separator between pivot tables

Pivot Table for TRIP_PURPOSE_AGG:
     DIST_PRODUCTION  DIST_ATTRACTION  HBO  HBW   NHB
0                  1                1  0.0  0.0  0.00
1                  1                2  0.0  0.0  0.00
2                  1                3  0.0  0.0  0.00
3                  1                4  0.0  0.0  0.00
4                  1                5  0.0  0.0  0.05
..               ...              ...  ...  ...   ...
895               30               26  0.0  0.0  0.00
896               30               27  0.0  0.0  0.00
897               30               28  0.0  0.0  0.00
898               30               29  0.0  0.0  0.00
899               30               30  0.0  0.0  0.00

[900 rows x 5 columns]

---

Pivot Table for AUTO_OWNERSHIP:
     DIST_PRODUCTION  DIST_ATTRACTION  None (0)  One (1)  Two or more (2+)
0                  1                1       0.0     0.00               0.0
1                  1                2       0.0     0.00               0.0
2                  1           

In [10]:
D2D_table_long_format


,DIST_PRODUCTION,DIST_ATTRACTION,KNR,PNR,Walk
0,1,1,0.0,0.0,0.00
1,1,2,0.0,0.0,0.00
2,1,3,0.0,0.0,0.00
3,1,4,0.0,0.0,0.00
4,1,5,0.0,0.0,0.05
...,...,...,...,...,...
895,30,26,0.0,0.0,0.00
896,30,27,0.0,0.0,0.00
897,30,28,0.0,0.0,0.00
898,30,29,0.0,0.0,0.00


In [11]:
# Convert the long format trip data into D2D format
D2D_table_new = D2D_table_long_format.pivot(index='DIST_PRODUCTION', columns='DIST_ATTRACTION', values='PNR')

D2D_table_new.reset_index(inplace=True)
D2D_table_new.columns.name = None
D2D_table_new.columns = [''] + list(D2D_table_new.columns[1:])
D2D_table_new.to_csv(output_dir + 'Table_Appendix_A_D2D_survey_trip_PNR.csv',index=False)


In [12]:
df.to_csv(interim_dir + 'Interim_2023_OnBoardSurvey_Dataset_3.csv',index=False)